# Data Preparation

This notebook prepares the raw NYC building violation dataset for
feature engineering and machine learning.

The raw dataset is kept unchanged. All preprocessing is performed
on a separate working copy.

In [1]:
import pandas as pd
import numpy as np

RAW_PATH = "../data/raw/DOB_ECB_Violations_20260913 (1).csv"

df_raw = pd.read_csv(RAW_PATH)

print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

Rows: 102467
Columns: 46


C:\Users\gimha_9pk7du7\AppData\Local\Temp\ipykernel_24388\3671810454.py:6: DtypeWarning: Columns (28: SECTION_LAW_DESCRIPTION3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(RAW_PATH)


In [2]:
df = df_raw.copy()

print("Raw dataset:", df_raw.shape)
print("Working dataset:", df.shape)

Raw dataset: (102467, 46)
Working dataset: (102467, 46)


In [3]:
# Remove confirmed identifier columns

identifier_columns = [
    "ISN_DOB_BIS_EXTRACT",
    "ECB_VIOLATION_NUMBER",
    "DOB_VIOLATION_NUMBER"
]

df = df.drop(columns=identifier_columns)

print("Removed columns:", identifier_columns)
print("Remaining rows:", df.shape[0])
print("Remaining columns:", df.shape[1])

Removed columns: ['ISN_DOB_BIS_EXTRACT', 'ECB_VIOLATION_NUMBER', 'DOB_VIOLATION_NUMBER']
Remaining rows: 102467
Remaining columns: 43


In [4]:
# Convert date columns to proper datetime format

date_columns = {
    "ISSUE_DATE": "ISSUE_DATE_DT",
    "SERVED_DATE": "SERVED_DATE_DT",
    "HEARING_DATE": "HEARING_DATE_DT"
}

for original_col, new_col in date_columns.items():
    df[new_col] = pd.to_datetime(
        df[original_col].astype(str),
        format="%Y%m%d",
        errors="coerce"
    )

print(df[
    [
        "ISSUE_DATE",
        "ISSUE_DATE_DT",
        "SERVED_DATE",
        "SERVED_DATE_DT",
        "HEARING_DATE",
        "HEARING_DATE_DT"
    ]
].head())

   ISSUE_DATE ISSUE_DATE_DT  SERVED_DATE SERVED_DATE_DT  HEARING_DATE  \
0    20260601    2026-06-01     20260601     2026-06-01      20260821   
1    20250805    2025-08-05     20250805     2025-08-05      20270415   
2    20260624    2026-06-24     20260624     2026-06-24      20260902   
3    20251203    2025-12-03     20251203     2025-12-03      20260908   
4    20260722    2026-07-22     20260819     2026-08-19      20261030   

  HEARING_DATE_DT  
0      2026-08-21  
1      2027-04-15  
2      2026-09-02  
3      2026-09-08  
4      2026-10-30  


In [5]:
for col in [
    "ISSUE_DATE_DT",
    "SERVED_DATE_DT",
    "HEARING_DATE_DT"
]:
    print(
        col,
        "| Missing:", df[col].isna().sum(),
        "| Min:", df[col].min(),
        "| Max:", df[col].max()
    )

ISSUE_DATE_DT | Missing: 0 | Min: 2025-01-01 00:00:00 | Max: 2026-09-10 00:00:00
SERVED_DATE_DT | Missing: 7 | Min: 2025-01-01 00:00:00 | Max: 2026-09-10 00:00:00
HEARING_DATE_DT | Missing: 0 | Min: 2025-02-06 00:00:00 | Max: 2027-12-22 00:00:00


In [6]:
# Create temporal features from ISSUE_DATE

df["ISSUE_YEAR"] = df["ISSUE_DATE_DT"].dt.year
df["ISSUE_MONTH"] = df["ISSUE_DATE_DT"].dt.month
df["ISSUE_DAY"] = df["ISSUE_DATE_DT"].dt.day
df["ISSUE_DAY_OF_WEEK"] = df["ISSUE_DATE_DT"].dt.dayofweek
df["ISSUE_QUARTER"] = df["ISSUE_DATE_DT"].dt.quarter

print(
    df[
        [
            "ISSUE_DATE_DT",
            "ISSUE_YEAR",
            "ISSUE_MONTH",
            "ISSUE_DAY",
            "ISSUE_DAY_OF_WEEK",
            "ISSUE_QUARTER"
        ]
    ].head()
)

  ISSUE_DATE_DT  ISSUE_YEAR  ISSUE_MONTH  ISSUE_DAY  ISSUE_DAY_OF_WEEK  \
0    2026-06-01        2026            6          1                  0   
1    2025-08-05        2025            8          5                  1   
2    2026-06-24        2026            6         24                  2   
3    2025-12-03        2025           12          3                  2   
4    2026-07-22        2026            7         22                  2   

   ISSUE_QUARTER  
0              2  
1              3  
2              2  
3              4  
4              3  


In [7]:
# Missing-value summary after date conversion and temporal feature creation

missing_summary = (
    df.isnull()
      .sum()
      .to_frame("Missing_Count")
)

missing_summary["Missing_Percentage"] = (
    missing_summary["Missing_Count"] / len(df) * 100
)

missing_summary = (
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Percentage", ascending=False)
)

print(missing_summary)

                           Missing_Count  Missing_Percentage
INFRACTION_CODE4                  102467          100.000000
INFRACTION_CODE7                  102467          100.000000
SECTION_LAW_DESCRIPTION10         102467          100.000000
INFRACTION_CODE10                 102467          100.000000
SECTION_LAW_DESCRIPTION9          102467          100.000000
INFRACTION_CODE9                  102467          100.000000
SECTION_LAW_DESCRIPTION8          102467          100.000000
INFRACTION_CODE8                  102467          100.000000
SECTION_LAW_DESCRIPTION7          102467          100.000000
INFRACTION_CODE5                  102467          100.000000
SECTION_LAW_DESCRIPTION6          102467          100.000000
INFRACTION_CODE6                  102467          100.000000
SECTION_LAW_DESCRIPTION5          102467          100.000000
SECTION_LAW_DESCRIPTION4          102467          100.000000
INFRACTION_CODE3                  102447           99.980482
SECTION_LAW_DESCRIPTION3

In [8]:
# Columns to remove from the working dataset

columns_to_remove = [
    # Empty / extremely sparse infraction fields
    "INFRACTION_CODE2",
    "SECTION_LAW_DESCRIPTION2",
    "INFRACTION_CODE3",
    "SECTION_LAW_DESCRIPTION3",
    "INFRACTION_CODE4",
    "SECTION_LAW_DESCRIPTION4",
    "INFRACTION_CODE5",
    "SECTION_LAW_DESCRIPTION5",
    "INFRACTION_CODE6",
    "SECTION_LAW_DESCRIPTION6",
    "INFRACTION_CODE7",
    "SECTION_LAW_DESCRIPTION7",
    "INFRACTION_CODE8",
    "SECTION_LAW_DESCRIPTION8",
    "INFRACTION_CODE9",
    "SECTION_LAW_DESCRIPTION9",
    "INFRACTION_CODE10",
    "SECTION_LAW_DESCRIPTION10",

    # Post-issue / potential leakage fields
    "SERVED_DATE",
    "SERVED_DATE_DT",
    "HEARING_DATE",
    "HEARING_DATE_DT",
    "HEARING_TIME",
    "ECB_VIOLATION_STATUS",
    "HEARING_STATUS",
    "CERTIFICATION_STATUS",

    # Financial fields that may reflect later case outcomes
    "PENALITY_IMPOSED",
    "AMOUNT_PAID",
    "BALANCE_DUE",

    # High-cardinality respondent/address fields
    "RESPONDENT_NAME",
    "RESPONDENT_HOUSE_NUMBER",
    "RESPONDENT_STREET"
]

# Check that all columns exist before removing them
missing_columns = [
    col for col in columns_to_remove
    if col not in df.columns
]

print("Columns requested for removal:", len(columns_to_remove))
print("Columns not found:", missing_columns)

df = df.drop(columns=columns_to_remove)

print("\nRemaining shape:", df.shape)

Columns requested for removal: 32
Columns not found: []

Remaining shape: (102467, 19)


In [9]:
print("Remaining columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2}. {col}")

Remaining columns:
 1. BIN
 2. BORO
 3. BLOCK
 4. LOT
 5. ISSUE_DATE
 6. SEVERITY
 7. VIOLATION_TYPE
 8. RESPONDENT_CITY
 9. RESPONDENT_ZIP
10. VIOLATION_DESCRIPTION
11. INFRACTION_CODE1
12. SECTION_LAW_DESCRIPTION1
13. AGGRAVATED_LEVEL
14. ISSUE_DATE_DT
15. ISSUE_YEAR
16. ISSUE_MONTH
17. ISSUE_DAY
18. ISSUE_DAY_OF_WEEK
19. ISSUE_QUARTER


In [10]:
missing_remaining = (
    df.isnull()
      .sum()
      .to_frame("Missing_Count")
)

missing_remaining["Missing_Percentage"] = (
    missing_remaining["Missing_Count"] / len(df) * 100
)

missing_remaining = (
    missing_remaining[missing_remaining["Missing_Count"] > 0]
    .sort_values("Missing_Percentage", ascending=False)
)

print(missing_remaining)

                          Missing_Count  Missing_Percentage
RESPONDENT_ZIP                     8932            8.716953
RESPONDENT_CITY                    8279            8.079674
SECTION_LAW_DESCRIPTION1           7994            7.801536
BLOCK                               487            0.475275
LOT                                 487            0.475275
BIN                                  95            0.092713
VIOLATION_DESCRIPTION                 8            0.007807


In [11]:
# ============================================================
# BUILDING HISTORY FEATURES
# ============================================================
# Count only violations that happened BEFORE the current
# ISSUE_DATE. Violations on the same date are NOT counted.

history = df[[
    "BIN",
    "ISSUE_DATE_DT",
    "SEVERITY"
]].copy()

# Count violations for each BIN on each issue date
daily_history = (
    history
    .groupby(["BIN", "ISSUE_DATE_DT"], dropna=False)
    .agg(
        violations_on_date=("SEVERITY", "size"),
        class1_on_date=("SEVERITY", lambda x: (x == "CLASS - 1").sum()),
        class2_on_date=("SEVERITY", lambda x: (x == "CLASS - 2").sum()),
        class3_on_date=("SEVERITY", lambda x: (x == "CLASS - 3").sum())
    )
    .reset_index()
)

# Calculate history BEFORE the current date
daily_history["PREVIOUS_VIOLATION_COUNT"] = (
    daily_history
    .groupby("BIN")["violations_on_date"]
    .cumsum()
    - daily_history["violations_on_date"]
)

daily_history["PREVIOUS_CLASS1_COUNT"] = (
    daily_history
    .groupby("BIN")["class1_on_date"]
    .cumsum()
    - daily_history["class1_on_date"]
)

daily_history["PREVIOUS_CLASS2_COUNT"] = (
    daily_history
    .groupby("BIN")["class2_on_date"]
    .cumsum()
    - daily_history["class2_on_date"]
)

daily_history["PREVIOUS_CLASS3_COUNT"] = (
    daily_history
    .groupby("BIN")["class3_on_date"]
    .cumsum()
    - daily_history["class3_on_date"]
)

# Keep only the features we need to merge back
history_features = daily_history[
    [
        "BIN",
        "ISSUE_DATE_DT",
        "PREVIOUS_VIOLATION_COUNT",
        "PREVIOUS_CLASS1_COUNT",
        "PREVIOUS_CLASS2_COUNT",
        "PREVIOUS_CLASS3_COUNT"
    ]
]

# Merge historical information back to every violation
df = df.merge(
    history_features,
    on=["BIN", "ISSUE_DATE_DT"],
    how="left"
)

# Missing BIN means there is no reliable building history
history_columns = [
    "PREVIOUS_VIOLATION_COUNT",
    "PREVIOUS_CLASS1_COUNT",
    "PREVIOUS_CLASS2_COUNT",
    "PREVIOUS_CLASS3_COUNT"
]

df[history_columns] = df[history_columns].fillna(0)

print("Historical features created:")
print(history_columns)

print("\nMissing values:")
print(df[history_columns].isna().sum())

print("\nStatistics:")
print(df[history_columns].describe())

Historical features created:
['PREVIOUS_VIOLATION_COUNT', 'PREVIOUS_CLASS1_COUNT', 'PREVIOUS_CLASS2_COUNT', 'PREVIOUS_CLASS3_COUNT']

Missing values:
PREVIOUS_VIOLATION_COUNT    0
PREVIOUS_CLASS1_COUNT       0
PREVIOUS_CLASS2_COUNT       0
PREVIOUS_CLASS3_COUNT       0
dtype: int64

Statistics:
       PREVIOUS_VIOLATION_COUNT  PREVIOUS_CLASS1_COUNT  PREVIOUS_CLASS2_COUNT  \
count             102467.000000          102467.000000          102467.000000   
mean                   1.669250               0.749217               0.901305   
std                    3.504407               1.991983               2.116281   
min                    0.000000               0.000000               0.000000   
25%                    0.000000               0.000000               0.000000   
50%                    0.000000               0.000000               0.000000   
75%                    2.000000               0.000000               1.000000   
max                   60.000000              32.000000  

In [12]:
# Handle missing categorical/text values

text_missing_columns = [
    "RESPONDENT_CITY",
    "RESPONDENT_ZIP",
    "SECTION_LAW_DESCRIPTION1",
    "VIOLATION_DESCRIPTION"
]

for col in text_missing_columns:
    df[col] = df[col].fillna("UNKNOWN")

print("Missing values after text handling:")

for col in text_missing_columns:
    print(
        f"{col}: {df[col].isna().sum()}"
    )

Missing values after text handling:
RESPONDENT_CITY: 0
RESPONDENT_ZIP: 0
SECTION_LAW_DESCRIPTION1: 0
VIOLATION_DESCRIPTION: 0


In [13]:
print("Dtypes:")
print(df[["BIN", "BLOCK", "LOT"]].dtypes)

print("\nMissing rows:")
print(df[["BIN", "BLOCK", "LOT"]].isna().sum())

print("\nRows where BIN is missing:")
print(
    df[df["BIN"].isna()][
        ["BIN", "BLOCK", "LOT", "BORO", "SEVERITY"]
    ].head(20)
)

Dtypes:
BIN      float64
BLOCK    float64
LOT      float64
dtype: object

Missing rows:
BIN       95
BLOCK    487
LOT      487
dtype: int64

Rows where BIN is missing:
      BIN  BLOCK  LOT  BORO   SEVERITY
5719  NaN    NaN  NaN     4  CLASS - 2
5735  NaN    NaN  NaN     3  CLASS - 2
5831  NaN    NaN  NaN     4  CLASS - 2
5866  NaN    NaN  NaN     2  CLASS - 2
5894  NaN    NaN  NaN     4  CLASS - 2
5900  NaN    NaN  NaN     3  CLASS - 1
5947  NaN    NaN  NaN     4  CLASS - 2
6009  NaN    NaN  NaN     4  CLASS - 2
6022  NaN    NaN  NaN     3  CLASS - 2
6123  NaN    NaN  NaN     3  CLASS - 2
6202  NaN    NaN  NaN     4  CLASS - 2
6220  NaN    NaN  NaN     5  CLASS - 1
6223  NaN    NaN  NaN     1  CLASS - 2
6235  NaN    NaN  NaN     5  CLASS - 2
6268  NaN    NaN  NaN     3  CLASS - 1
6295  NaN    NaN  NaN     4  CLASS - 2
6321  NaN    NaN  NaN     4  CLASS - 2
6364  NaN    NaN  NaN     2  CLASS - 2
6380  NaN    NaN  NaN     5  CLASS - 2
6416  NaN    NaN  NaN     4  CLASS - 2


In [14]:
# Handle missing building/location identifiers

building_columns = ["BIN", "BLOCK", "LOT"]

for col in building_columns:
    df[col] = df[col].astype("string").fillna("UNKNOWN")

print("Missing values after handling:")

print(df[building_columns].isna().sum())

print("\nSample values:")
print(df[building_columns].head(10))

print("\nDtypes:")
print(df[building_columns].dtypes)

Missing values after handling:
BIN      0
BLOCK    0
LOT      0
dtype: int64

Sample values:
         BIN   BLOCK    LOT
0  2026562.0  3813.0   19.0
1  1003388.0   279.0   69.0
2  5020516.0   795.0   41.0
3  4139680.0  6381.0   54.0
4  2016355.0  3277.0    2.0
5  3014376.0   817.0   46.0
6  3331428.0  8329.0  225.0
7  1063300.0  2135.0   60.0
8  5106881.0   565.0   46.0
9  3123714.0  5308.0    6.0

Dtypes:
BIN      string
BLOCK    string
LOT      string
dtype: object


In [15]:
print("=" * 60)
print("FINAL PREPARATION AUDIT")
print("=" * 60)

print("\nShape:")
print(df.shape)

print("\nMissing values:")
missing = df.isnull().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("No missing values remaining.")
else:
    print(missing)

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nTarget distribution:")
print(df["SEVERITY"].value_counts())

print("\nTarget percentage:")
print(
    df["SEVERITY"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nData types:")
print(df.dtypes)

print("\nHistorical features:")
history_cols = [
    "PREVIOUS_VIOLATION_COUNT",
    "PREVIOUS_CLASS1_COUNT",
    "PREVIOUS_CLASS2_COUNT",
    "PREVIOUS_CLASS3_COUNT"
]

print(df[history_cols].describe().T)

FINAL PREPARATION AUDIT

Shape:
(102467, 23)

Missing values:
No missing values remaining.

Duplicate rows:
508

Target distribution:
SEVERITY
CLASS - 2    57607
CLASS - 1    40673
CLASS - 3     4187
Name: count, dtype: int64

Target percentage:
SEVERITY
CLASS - 2    56.22
CLASS - 1    39.69
CLASS - 3     4.09
Name: proportion, dtype: float64

Data types:
BIN                                 string
BORO                                 int64
BLOCK                               string
LOT                                 string
ISSUE_DATE                           int64
SEVERITY                               str
VIOLATION_TYPE                         str
RESPONDENT_CITY                        str
RESPONDENT_ZIP                         str
VIOLATION_DESCRIPTION                  str
INFRACTION_CODE1                       str
SECTION_LAW_DESCRIPTION1               str
AGGRAVATED_LEVEL                       str
ISSUE_DATE_DT               datetime64[us]
ISSUE_YEAR                           int

In [16]:
duplicate_rows = df[df.duplicated(keep=False)].copy()

print("Duplicate rows involved:", len(duplicate_rows))

print("\nNumber of duplicate groups:")
print(
    duplicate_rows
    .groupby(list(df.columns))
    .size()
    .value_counts()
    .sort_index()
)

print("\nSample duplicate records:")
display(
    duplicate_rows.sort_values(
        ["BIN", "ISSUE_DATE_DT"]
    ).head(20)
)

Duplicate rows involved: 827

Number of duplicate groups:
2     224
3      51
4      22
5       9
6       7
7       2
8       2
10      1
11      1
Name: count, dtype: int64

Sample duplicate records:


,BIN,BORO,BLOCK,LOT,ISSUE_DATE,SEVERITY,VIOLATION_TYPE,RESPONDENT_CITY,RESPONDENT_ZIP,VIOLATION_DESCRIPTION,...,ISSUE_DATE_DT,ISSUE_YEAR,ISSUE_MONTH,ISSUE_DAY,ISSUE_DAY_OF_WEEK,ISSUE_QUARTER,PREVIOUS_VIOLATION_COUNT,PREVIOUS_CLASS1_COUNT,PREVIOUS_CLASS2_COUNT,PREVIOUS_CLASS3_COUNT
17939,1000807.0,1,20.0,16.0,20250219,CLASS - 1,Elevators,NEW YORK,23510,UPON INSPECTION OF ESCALATORS THERE ISNO SERVI...,...,2025-02-19,2025,2,19,2,1,0.0,0.0,0.0,0.0
18860,1000807.0,1,20.0,16.0,20250219,CLASS - 1,Elevators,NEW YORK,23510,UPON INSPECTION OF ESCALATORS THERE ISNO SERVI...,...,2025-02-19,2025,2,19,2,1,0.0,0.0,0.0,0.0
72215,1000807.0,1,20.0,16.0,20250425,CLASS - 1,Elevators,NORFOLK,23510,CLASS 1:FAILURE TO MAINTAIN BUILDING IN CODE C...,...,2025-04-25,2025,4,25,4,2,2.0,2.0,0.0,0.0
72392,1000807.0,1,20.0,16.0,20250425,CLASS - 1,Elevators,NORFOLK,23510,CLASS 1:FAILURE TO MAINTAIN BUILDING IN CODE C...,...,2025-04-25,2025,4,25,4,2,2.0,2.0,0.0,0.0
46070,1001006.0,1,40.0,16.0,20250630,CLASS - 1,Elevators,NEW YORK,10016,CLASS 1. RETURN TO SERVICE. : FAILURE TO MAINT...,...,2025-06-30,2025,6,30,0,2,0.0,0.0,0.0,0.0
47143,1001006.0,1,40.0,16.0,20250630,CLASS - 1,Elevators,NEW YORK,10016,CLASS 1. RETURN TO SERVICE. : FAILURE TO MAINT...,...,2025-06-30,2025,6,30,0,2,0.0,0.0,0.0,0.0
19656,1001006.0,1,40.0,16.0,20260709,CLASS - 2,Elevators,NEW YORK,10005,CLASS 2 RTS: ELEVATOR DEVICE IN A MULTI-ELEVAT...,...,2026-07-09,2026,7,9,3,3,8.0,6.0,2.0,0.0
20556,1001006.0,1,40.0,16.0,20260709,CLASS - 2,Elevators,NEW YORK,10005,CLASS 2 RTS: ELEVATOR DEVICE IN A MULTI-ELEVAT...,...,2026-07-09,2026,7,9,3,3,8.0,6.0,2.0,0.0
45280,1003269.0,1,274.0,14.0,20260115,CLASS - 2,Construction,GREAT NECK,11021,FAILURE TO COMPLY WITH COMMISSIONER'S ORDER TO...,...,2026-01-15,2026,1,15,3,1,4.0,4.0,0.0,0.0
48238,1003269.0,1,274.0,14.0,20260115,CLASS - 2,Construction,GREAT NECK,11021,FAILURE TO COMPLY WITH COMMISSIONER'S ORDER TO...,...,2026-01-15,2026,1,15,3,1,4.0,4.0,0.0,0.0


In [17]:
print("DataFrames currently in memory:")

for name, obj in list(globals().items()):
    if hasattr(obj, "shape") and hasattr(obj, "columns"):
        print(f"{name}: {obj.shape}")

DataFrames currently in memory:
df_raw: (102467, 46)
df: (102467, 23)
missing_summary: (31, 2)
missing_remaining: (7, 2)
history: (102467, 3)
daily_history: (62999, 10)
history_features: (62999, 6)
duplicate_rows: (827, 23)


In [18]:
# Check whether the rows that look duplicated in df
# actually have different original violation identifiers.

duplicate_indices = duplicate_rows.index

original_duplicate_info = df_raw.loc[
    duplicate_indices,
    [
        "ISN_DOB_BIS_EXTRACT",
        "ECB_VIOLATION_NUMBER",
        "DOB_VIOLATION_NUMBER",
        "BIN",
        "ISSUE_DATE",
        "SEVERITY"
    ]
]

print("Original identifiers for duplicate-looking rows:")
display(original_duplicate_info.head(30))

print("\nUnique ECB violation numbers:")
print(original_duplicate_info["ECB_VIOLATION_NUMBER"].nunique())

print("\nUnique ISN records:")
print(original_duplicate_info["ISN_DOB_BIS_EXTRACT"].nunique())

Original identifiers for duplicate-looking rows:


,ISN_DOB_BIS_EXTRACT,ECB_VIOLATION_NUMBER,DOB_VIOLATION_NUMBER,BIN,ISSUE_DATE,SEVERITY
149,1367443,35688337L,03182025CMTFRF07,4539502.0,20250318,CLASS - 1
453,1724715,39165517L,NaN,3345765.0,20251031,CLASS - 2
824,1698417,39173077R,NaN,2014845.0,20260108,CLASS - 1
968,1770330,39193733Y,NaN,3418180.0,20260625,CLASS - 2
999,1367342,35688331K,03182025CMTFRF01,4539502.0,20250318,CLASS - 1
1026,1367345,35688332M,03182025CMTFRF02,4539502.0,20250318,CLASS - 1
1359,1367423,35688336J,03182025CMTFRF06,4539502.0,20250318,CLASS - 1
1364,1631059,39142488J,NaN,4035427.0,20250321,CLASS - 2
1478,1459287,39148175X,NaN,3430256.0,20250516,CLASS - 1
1716,1461265,39148204X,NaN,3430256.0,20250516,CLASS - 1



Unique ECB violation numbers:
827

Unique ISN records:
827


In [19]:
print("AGGRAVATED_LEVEL values:")
print(df_raw["AGGRAVATED_LEVEL"].value_counts(dropna=False))

print("\nAGGRAVATED_LEVEL by ISSUE_DATE:")
print(
    df_raw.groupby("AGGRAVATED_LEVEL")["ISSUE_DATE"]
    .agg(["min", "max", "count"])
)

print("\nAGGRAVATED_LEVEL by HEARING_DATE:")
print(
    df_raw.groupby("AGGRAVATED_LEVEL")["HEARING_DATE"]
    .agg(["min", "max", "count"])
)

AGGRAVATED_LEVEL values:
AGGRAVATED_LEVEL
NO                            94199
AGGRAVATED OFFENSE LEVEL 1     7323
AGGRAVATED OFFENSE LEVEL 2      945
Name: count, dtype: int64

AGGRAVATED_LEVEL by ISSUE_DATE:
                                 min       max  count
AGGRAVATED_LEVEL                                     
AGGRAVATED OFFENSE LEVEL 1  20250102  20260908   7323
AGGRAVATED OFFENSE LEVEL 2  20250102  20260902    945
NO                          20250101  20260910  94199

AGGRAVATED_LEVEL by HEARING_DATE:
                                 min       max  count
AGGRAVATED_LEVEL                                     
AGGRAVATED OFFENSE LEVEL 1  20250318  20271222   7323
AGGRAVATED OFFENSE LEVEL 2  20250318  20270610    945
NO                          20250206  20271015  94199


In [20]:
# ============================================================
# FINAL MODELING DATASET
# ============================================================

model_df = df.copy()

# AGGRAVATED_LEVEL is excluded because its timing relative
# to the prediction point cannot be established safely.
model_df = model_df.drop(columns=["AGGRAVATED_LEVEL"])

print("Modeling dataset shape:", model_df.shape)

print("\nColumns:")
for i, col in enumerate(model_df.columns, start=1):
    print(f"{i:2}. {col}")

Modeling dataset shape: (102467, 22)

Columns:
 1. BIN
 2. BORO
 3. BLOCK
 4. LOT
 5. ISSUE_DATE
 6. SEVERITY
 7. VIOLATION_TYPE
 8. RESPONDENT_CITY
 9. RESPONDENT_ZIP
10. VIOLATION_DESCRIPTION
11. INFRACTION_CODE1
12. SECTION_LAW_DESCRIPTION1
13. ISSUE_DATE_DT
14. ISSUE_YEAR
15. ISSUE_MONTH
16. ISSUE_DAY
17. ISSUE_DAY_OF_WEEK
18. ISSUE_QUARTER
19. PREVIOUS_VIOLATION_COUNT
20. PREVIOUS_CLASS1_COUNT
21. PREVIOUS_CLASS2_COUNT
22. PREVIOUS_CLASS3_COUNT


In [21]:
# ============================================================
# CHECK TEMPORAL TRAIN / TEST SPLIT
# ============================================================

model_df = model_df.sort_values("ISSUE_DATE_DT").copy()

split_index = int(len(model_df) * 0.80)

split_date = model_df.iloc[split_index]["ISSUE_DATE_DT"]

print("Total records:", len(model_df))
print("80% split index:", split_index)
print("Proposed split date:", split_date)

print("\nDate range:")
print("Start:", model_df["ISSUE_DATE_DT"].min())
print("End:  ", model_df["ISSUE_DATE_DT"].max())

print("\nRecords before split:")
print((model_df["ISSUE_DATE_DT"] < split_date).sum())

print("Records on/after split:")
print((model_df["ISSUE_DATE_DT"] >= split_date).sum())

print("\nTarget distribution before split:")
print(
    model_df.loc[
        model_df["ISSUE_DATE_DT"] < split_date,
        "SEVERITY"
    ].value_counts(normalize=True).mul(100).round(2)
)

print("\nTarget distribution after split:")
print(
    model_df.loc[
        model_df["ISSUE_DATE_DT"] >= split_date,
        "SEVERITY"
    ].value_counts(normalize=True).mul(100).round(2)
)

Total records: 102467
80% split index: 81973
Proposed split date: 2026-05-13 00:00:00

Date range:
Start: 2025-01-01 00:00:00
End:   2026-09-10 00:00:00

Records before split:
81903
Records on/after split:
20564

Target distribution before split:
SEVERITY
CLASS - 2    56.55
CLASS - 1    39.33
CLASS - 3     4.11
Name: proportion, dtype: float64

Target distribution after split:
SEVERITY
CLASS - 2    54.89
CLASS - 1    41.14
CLASS - 3     3.97
Name: proportion, dtype: float64


In [22]:
# ============================================================
# SAVE CLEANED MODELING DATASET
# ============================================================

output_path = "../data/processed/buildsafe_cleaned.csv"

model_df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print("Path:", output_path)
print("Shape:", model_df.shape)

Cleaned dataset saved successfully.
Path: ../data/processed/buildsafe_cleaned.csv
Shape: (102467, 22)
